# PaySim exploratory data analysis
Run only against a locally downloaded dataset. The CSV is intentionally not bundled. Do not interpret synthetic demo data as evidence.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path('../data/PS_20174393_ fraudDetection.csv')  # edit to your downloaded filename
assert DATA.exists(), f'Place the downloaded CSV at {DATA}'
df = pd.read_csv(DATA)
df.shape, df.columns.tolist()

## Data quality and class balance
Check missing values, duplicates, class prevalence, and label distribution before modelling.

In [ ]:
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame('missing'))
display(df['isFraud'].value_counts(dropna=False).to_frame('rows'))
print('Fraud prevalence:', float(df['isFraud'].mean()))
print('Duplicate rows:', int(df.duplicated().sum()))

## Fraud by transaction type

In [ ]:
type_summary = df.groupby('type').agg(transactions=('isFraud','size'), frauds=('isFraud','sum'), fraud_rate=('isFraud','mean')).sort_values('fraud_rate', ascending=False)
display(type_summary)
type_summary['fraud_rate'].plot(kind='bar', title='Fraud rate by transaction type', ylabel='Fraud rate')
plt.tight_layout()

## Amount distribution
Use log scale because transaction amounts are typically highly skewed.

In [ ]:
df.assign(log_amount=(df['amount'].clip(lower=0)+1).map(__import__('math').log10)).groupby('isFraud')['log_amount'].plot(kind='hist', bins=50, alpha=.5, legend=True)
plt.title('Log10(amount + 1) by fraud label')
plt.xlabel('Log10(amount + 1)')
plt.tight_layout()

## Fraud by simulated hour
`step` is the dataset's simulated time index; confirm dataset documentation before interpreting it as elapsed clock time.

In [ ]:
hourly = df.groupby('step').agg(transactions=('isFraud','size'), frauds=('isFraud','sum'))
hourly['fraud_rate'] = hourly['frauds'] / hourly['transactions']
display(hourly.head())
hourly['fraud_rate'].plot(title='Fraud rate by PaySim step', ylabel='Fraud rate')
plt.tight_layout()

## Interpretation checklist
- Separate exploratory observations from causal claims.
- Report absolute counts alongside rates because fraud is rare.
- Check whether labels or post-transaction balance fields leak the target into a model.
- Document dataset version, hash, exclusions, and code environment.
- Never publish demo-generated metric gaps as findings.